# Batching AI calls

When each call has fixed overhead, batching wins. We simulate it.

In [ ]:
import asyncio, time

PER_CALL_BASE = 0.1   # overhead on each request
PER_ITEM_COST = 0.005 # cost per item inside a call

async def call(items):
    await asyncio.sleep(PER_CALL_BASE + PER_ITEM_COST * len(items))
    return [i*i for i in items]

async def one_by_one(items):
    return await asyncio.gather(*(call([i]) for i in items))

async def batched(items, b):
    chunks = [items[i:i+b] for i in range(0, len(items), b)]
    return await asyncio.gather(*(call(c) for c in chunks))

items = list(range(50))
for label, fn in [
    ('one_by_one', lambda: one_by_one(items)),
    ('batch=5',    lambda: batched(items, 5)),
    ('batch=10',   lambda: batched(items, 10)),
    ('batch=50',   lambda: batched(items, 50)),
]:
    t = time.perf_counter()
    await fn()
    print(f'{label:12} ms={(time.perf_counter()-t)*1000:7.1f}')

## Reflect

- The per-call overhead dominates without batching. Real providers behave the same way.
- The batched version is also cheaper $-wise since you pay per request.